# RAG Pipeline Architecture

Architecture and progress summary for Assignment 2B. Parts A and B are complete; Parts C and D remain to be implemented. The executable retrieval evaluation and its detailed results remain in `assignment_2b_rag.ipynb` and `results/`.

## Pipeline Diagram

```mermaid
flowchart LR
    corpus["AWS corpus<br/>6 cleaned documents"] --> chunks["Token-exact chunking<br/>100 vs 400; selected 400"]
    queries["10 C3 queries<br/>expected answers + exact spans"] --> eval["Shared hit@k evaluation"]
    chunks --> dense["Dense retrieval<br/>BGE-small 384-d<br/>normalized vectors + FAISS IndexFlatIP"]
    chunks --> bm25["Keyword retrieval<br/>BM25Okapi"]

    subgraph completed["Completed: Parts A and B"]
        chunks
        dense
        bm25
        dense --> compare["Part B comparison<br/>top 5 per method"]
        dense --> rrf["RRF fusion<br/>top 20 per method; k=60"]
        bm25 --> compare
        bm25 --> rrf
        rrf --> compare
        compare --> eval
        dense --> eval
        compare --> winner["Best measured method<br/>BM25"]
    end

    subgraph upcoming["To complete: Parts C and D"]
        top10["Part C: retrieve top 10<br/>with selected Part B method"]
        rerank["Cross-encoder reranking<br/>top 10 -> top 3"]
        answer["Part D: Mistral-7B-v0.1<br/>QLoRA adapter; cite retrieved chunks"]
        judge["Independent LLM judge<br/>correctness + evidence support"]
        spotcheck["Spot-check, failure analysis<br/>and not-found query"]
        top10 --> rerank --> answer --> judge --> spotcheck
    end

    winner --> top10
    queries --> top10
```

## Completed Work

### Part A: Dense Retrieval and Chunk Selection
- Curated an expected answer, exact supporting corpus span, and source filename for each of the ten C3 queries.
- Compared fixed, non-overlapping 100-token and 400-token chunks with `BAAI/bge-small-en-v1.5`, normalized embeddings, and FAISS `IndexFlatIP`.
- Verified the embedding model's 512-token input limit. The longest 400-token chunk was 403 tokens including special tokens, so no truncation was required.
- The 100-token set had 1,837 chunks, 0% hit@3, and 300 average top-three chunk tokens per query. The 400-token set had 461 chunks, 20% hit@3, and 1,200 average top-three tokens. The 400-token size was selected for subsequent parts.

### Part B: Keyword and Hybrid Retrieval
- Compared Dense, BM25, and reciprocal-rank fusion using the same ten queries and selected 400-token chunks. Each method returned five results for scoring; RRF fused the top 20 from each retriever using `1/(60 + rank)`. One warm-up run per method was excluded from latency.
- BM25 achieved the best measured hit rates: 50% hit@1 and 80% hit@3 at 5.59 ms average query latency. Dense retrieval achieved 0% hit@1 and 20% hit@3 at 74.26 ms; Hybrid RRF achieved 10% hit@1 and 40% hit@3 at 62.06 ms.
- For `c3_01_ec2_ssh`, BM25 retrieved the answer span in its top three while dense retrieval did not.
- Detailed timings, hit flags, and top-five chunk IDs are saved in `results/part_b_retrieval_comparison.json`; Part A metrics are in `results/part_a_chunk_comparison.json`.

## Remaining Work

### Part C: Reranking
Use the best Part B method (BM25 on current measurements) to retrieve ten candidates per query, rerank them with a cross-encoder, and retain the top three. Compare before/after rankings and report rank-change rate, reranking latency, and hit@1.

### Part D: Grounded Answers and Evaluation
Give the top-three reranked chunks to the copied Mistral-7B-v0.1 QLoRA adapter. Require citations to chunk numbers and a not-found response when the context lacks the answer. Use a different LLM to judge answer correctness and evidence support, then complete the spot-check, failure analysis, and out-of-corpus not-found example.